<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
موضوع‌بندی
</font>
</h1>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
مقدمه و صورت مسئله
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
به یک تمرین صنعتی از  کاربرد یادگیری ماشین در پردازش زبان طبیعی (NLP) خوش آمدید.  در این تمرین داده‌های واقعی وب فارسی که توسط پلتفرم <a href="https://www.yektanet.com/">یکتانت</a> پالایش و جمع‌آوری شده در اختیار ما قرار گرفته است. هدف تمرین؛ ساخت یک مدل یادگیری ماشین است که با توجه متن‌های موجود در یک پیوند (Link) نظیر <i>عنوان</i>، <i>توضیحات</i>، <i>محتوای متنی [کامل]</i> و غیره بتواند دسته‌ی موضوعی آن سند را پیش‌بینی کند. به‌عنوان مثال اگر پیوندی از یک سایت خبری با عنوان «<i>کیهان کلهر جایزه مرد سال موسیقی جهان را دریافت کرد</i>» داشته باشیم، مدل ما باید پیش‌بینی کند که این مطلب مرتبط با موضوع «موسیقی» است. البته در این مثال ما تنها از ویژگی <i>عنوان</i> یاد کردیم، در حالی‌که می‌توان از متن <i>توضیحات</i> یا <i>محتوای متنی</i> هم کمک گرفت.

</br>
توجه داشته باشید برای آن‌که بتوانید از الگوریتم‌های معرفی‌شده در کالج جهت کار روی داده‌های متنی استفاده کنید نیاز است حداقل با یکی از روش‌های تعبیه‌سازی (Embedding) آشنا شده باشید تا بتوانید هر متن را به یک بردار ویژگی عددی تبدیل کنید.

</font>
</p>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
وارد کردن کتابخانه‌های مورد نیاز
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    ابتدا کتابخانه‌های مورد نیازتان را وارد کنید.
</font>
</p>

In [ ]:
!pip install jupyter==1.1.1 jupyterlab==4.3.6 flaml==2.3.4 xgboost==3.0.0 scikit-learn==1.6.1 rbo==0.1.3 numpy==1.26.4 pandas==2.2.3 matplotlib==3.10.1 seaborn==0.13.2 pickle-mixin==1.0.2 joblib==1.4.2 parsivar plotly

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
import joblib

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
معرفی مجموعه‌داده
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    هر نمونه از این مجموعه‌داده با ویژگی‌هایی که در جدول زیر شرح داده شده همراه است. ستون <code>category</code> متغیر هدف مسئله است که موضوع محتوا را نشان می‌دهد.
</font>
</p>
<center>
<div dir=rtl style="direction: rtl;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    
|ستون|توضیحات|
|:------:|:---:|
|<code>category</code>| موضوع (متغیر هدف) |
|<code>description</code>| توضیحات |
|<code>text_content</code>| محتوای متنی |
|<code>title</code>| عنوان |
|<code>h1</code>| محتوای تگ <code>h1</code> صفحه |
|<code>h2</code>|محتوای تگ <code>h2</code> صفحه  |
|<code>url</code>| آدرس پیوند|
|<code>domain</code>|دامنه‌ی وب‌سایت |
|<code>id</code>|آیدی پیوند|

</font>
</div>
</center>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
خواندن مجموعه داده
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    در ابتدا نیاز است فایل‌های مجموعه‌داده را بخوانید. نمونه‌های آموزشی در فایل <code>yektanet_train.csv</code> و نمونه‌های آزمون که باید موضوع آن‌ها را پیش‌بینی کنید در فایل <code>yektanet_test.csv</code> ذخیره شده‌اند. اگر لازم دانستید می‌توانید به دلخواه خود بخشی از مجموعه‌ی آموزشی را به عنوان مجموعه‌ی اعتبارسنجی نیز جدا کنید.
</font>
</p>

In [ ]:
train = None # To-Do
train.head()

In [ ]:
test = None # To-Do
test.head()

In [ ]:
# خواندن فایل‌های داده
train = pd.read_csv('yektanet_train.csv')
test = pd.read_csv('yektanet_test.csv')

# بررسی شکل داده‌ها
print(f"تعداد نمونه‌های آموزشی: {train.shape[0]}")
print(f"تعداد نمونه‌های آزمون: {test.shape[0]}")
print(f"تعداد ویژگی‌ها: {train.shape[1]}")

# نمایش چند نمونه از داده‌ها
train.head()

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
پیش‌پردازش و مهندسی ویژگی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
در هنگام کار با داده‌های متنی، پیش‌پردازش داده‌ها به کمک تکنیک‌های پردازش زبان طبیعی یکی از مراحل بسیار تاثیرگذار در یادگیری مدل و عملکرد نهایی است.
در تمرین «کامنت‌کاوی» فصل دسته‌بندی با چندین تکنیک رایج پیش‌پردازش خصوصاً برای زبان فارسی آشنا شدید. در این قسمت نیز می‌توانید تابعی بنویسید که یک رشته (<code>string</code>) را گرفته، اصلاحات موردنظر شما رو روی متن اعمال کرده و در نهایت نتیجه را با فرمت یک رشته (<code>string</code>) خروجی دهد. بررسی و تحلیل کلمات موجود در مجموعه‌داده از نظر تعداد رخداد نیز می‌تواند شما را در پیش‌پردازش بهتر یاری کند.
</font>
</p>


In [ ]:
# To-Do

In [ ]:
# بررسی مقادیر خالی
print("مقادیر خالی در داده‌های آموزشی:")
print(train.isnull().sum())
print("\nمقادیر خالی در داده‌های آزمون:")
print(test.isnull().sum())

# پر کردن مقادیر خالی با رشته خالی
train = train.fillna('')
test = test.fillna('')

# بررسی توزیع کلاس‌ها
print("\nتوزیع کلاس‌ها:")
print(train['category'].value_counts())

In [ ]:
def combine_features(row):
    """ترکیب ویژگی‌های متنی مختلف"""
    # ترکیب تمام ویژگی‌های متنی با وزن‌دهی مناسب
    combined = ''

    # عنوان با اهمیت بیشتر (3 بار تکرار)
    if pd.notna(row['title']):
        combined += (row['title'] + ' ') * 3

    # توضیحات با اهمیت متوسط (2 بار تکرار)
    if pd.notna(row['description']):
        combined += (row['description'] + ' ') * 2

    # محتوای h1 و h2
    if pd.notna(row['h1']):
        combined += row['h1'] + ' '
    if pd.notna(row['h2']):
        combined += row['h2'] + ' '

    # محتوای متنی کامل
    if pd.notna(row['text_content']):
        combined += row['text_content']

    return combined.strip()

# اعمال ترکیب ویژگی‌ها
train['combined_text'] = train.apply(combine_features, axis=1)
test['combined_text'] = test.apply(combine_features, axis=1)

In [ ]:
import re
from hazm import Normalizer, word_tokenize, stopwords_list

# ایجاد نرمال‌ساز
normalizer = Normalizer()

# لیست کلمات توقف
stop_words = set(stopwords_list())
# اضافه کردن کلمات توقف اضافی
additional_stops = {'است', 'بود', 'شد', 'شده', 'می', 'کند', 'کرد', 'کرده', 'باشد', 'هستند'}
stop_words.update(additional_stops)

def preprocess_text(text):
    """پیش‌پردازش متن فارسی"""
    if pd.isna(text):
        return ''

    # نرمال‌سازی
    text = normalizer.normalize(text)

    # حذف اعداد انگلیسی
    text = re.sub(r'\d+', '', text)

    # حذف علائم نگارشی
    text = re.sub(r'[^\w\s]', ' ', text)

    # حذف فاصله‌های اضافی
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# اعمال پیش‌پردازش
train['processed_text'] = train['combined_text'].apply(preprocess_text)
test['processed_text'] = test['combined_text'].apply(preprocess_text)

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
در هنگام تعبیه‌سازی به کمک کتابخانه‌ی <code>scikit-learn</code>
از آنجا که ممکن است توکن‌ساز (Tokenizer) مورد استفاده در این کتابخانه مناسب زبان فارسی نباشد، بهتر است از توکن‌سازهای مناسب این زبان استفاده کنیم. کافیست تابعی بنویسید که یک رشته (<code>string</code>) را گرفته و به کمک کتابخانه‌ی موردنظر شما (مثل <code>word_tokenize</code> در کتابخانه‌ی هضم) توکن‌های آن را جدا کند. خروجی تابع لیستی از توکن‌ها خواهد بود.
</font>
</p>


In [ ]:
def tokenizer(text):
    return None # To-Do

In [ ]:
def tokenizer(text):
    """توکنایزر سفارشی برای متن فارسی"""
    if not text:
        return []

    # توکنایز کردن
    tokens = word_tokenize(text)

    # حذف کلمات توقف و کلمات کوتاه
    tokens = [token for token in tokens if token not in stop_words and len(token) > 1]

    return tokens

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
توجه داشته باشید انتخاب این‌که از کدام ویژگی‌ها (متن‌ها) به‌عنوان ورودی الگوریتم استفاده کنید بر عهده‌ی خودتان است. می‌توانید تنها از یک ستون مثل <code>title</code> استفاده کنید یا می‌توانید به‌نحوی متن یا حتی بردار ویژگی هر ستون را با همدیگر ترکیب کنید. همچنین فراموش نکنید که ستون متغیر هدف یعنی <code>category</code> نیاز به کدگذاری دارد.
</font>
</p>


In [ ]:
from sklearn.preprocessing import LabelEncoder

# To-Do

In [ ]:
from sklearn.preprocessing import LabelEncoder

# کدگذاری برچسب‌ها
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train['category'])

# ذخیره کلاس‌ها برای استفاده بعدی
classes = label_encoder.classes_
print(f"تعداد کلاس‌ها: {len(classes)}")
print(f"کلاس‌ها: {classes}")

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
مدل‌سازی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
اکنون وقت آن رسیده که الگوریتم یادگیری ماشین موردنظر خود را بر روی داده‌های آموزشی اجرا کنید. در انتخاب الگوریتم کاملاً آزاد هستید. برای این بخش می‌توانید ابتدا هر ورودی متن را به کمک تکنیک‌های معرفی‌شده در درسنامه‌های این فصل (مثل Bag-of-Word یا Tf-idf) به بردارهای ویژگی عددی تبدیل کنید. سپس این بردارها را همراه با لیست برچسب‌های صحیح به الگوریتم بدهید تا مدل آموخته شود. علاوه بر این می‌توانید تمام این مراحل را در یک خط لوله‌ (<code>Pipeline</code>) نیز تعریف کنید.
</font>
</p>

In [ ]:
# To-Do

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# آماده‌سازی داده‌ها
X_train = train['processed_text']
X_test = test['processed_text']

# ایجاد pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        tokenizer=tokenizer,
        max_features=10000,
        ngram_range=(1, 3),  # استفاده از unigram, bigram و trigram
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ('classifier', LinearSVC(
        C=1.0,
        class_weight='balanced',
        random_state=42,
        max_iter=5000
    ))
])

# آموزش مدل
pipeline.fit(X_train, y_train)

# ارزیابی با cross-validation
scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1_weighted')
print(f"میانگین F1-Score: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
معیار ارزیابی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    معیاری که برای ارزیابی عملکرد مدل انتخاب کرده‌ایم، <code>F1-score</code> نام دارد و از رویکرد میانگین‌گیری وزن‌دار (<code dir=ltr>average='weighted'</code>) استفاده می‌شود.
    <br>
    پیشنهاد می‌شود با توجه به این معیار، عملکرد مدل خود را بر روی مجموعه‌ی آموزش یا اعتبارسنجی ارزیابی کنید و طبق نتایج به‌دست‌آمده پارامترهای مدل خود را بهتر تنظیم کنید.
</font>
</p>

In [ ]:
from sklearn.metrics import f1_score

# To-Do

In [ ]:
from sklearn.model_selection import GridSearchCV

# تعریف پارامترها برای جستجو
param_grid = {
    'tfidf__max_features': [5000, 10000, 15000],
    'tfidf__ngram_range': [(1, 2), (1, 3)],
    'classifier__C': [0.1, 1.0, 10.0]
}

# جستجوی شبکه‌ای
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

# آموزش با بهترین پارامترها
grid_search.fit(X_train, y_train)

print(f"بهترین پارامترها: {grid_search.best_params_}")
print(f"بهترین F1-Score: {grid_search.best_score_:.4f}")

# استفاده از بهترین مدل
best_model = grid_search.best_estimator_

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
 پیش‌بینی برای داده تست و خروجی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    پس از مهندسی ویژگی و مدل‌سازی، الگوریتمی دارید که می‌تواند شما را از متغیرهای مستقل به متغیر هدف برساند.
    <br>
    از این مدل برای پیش‌بینی نمونه‌های موجود در مجموعه‌ی آزمون استفاده کنید و نتایج را در یک دیتافریم تک‌ستونه با نام <code>submission</code> و در قالب زیر آماده کنید. توجه داشته باشید که ترتیب پیش‌بینی شما اهمیت دارد یعنی به عنوان مثال پیش‌بینی مدل برای نمونه‌ی آزمون <code>m</code> ام را باید در ردیف <code>m</code> ام این دیتافریم ذخیره کنید.
</font>
</p>

<center>
<div dir=rtl style="direction: rtl;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    
|ستون|توضیحات|
|:------:|:---:|
|<code>category</code>|پیش‌بینی مدل شما (از جنس رشته)|
    
</font>
</div>
</center>

In [ ]:
submission = None # To-Do
submission.head()

In [ ]:
# پیش‌بینی برای داده‌های آزمون با استفاده از pipeline آموزش‌دیده
y_pred = pipeline.predict(X_test)

# تبدیل به برچسب‌های اصلی
predictions = label_encoder.inverse_transform(y_pred)

# ایجاد DataFrame خروجی
submission = pd.DataFrame({
    'category': predictions
})

# نمایش چند نمونه از پیش‌بینی‌ها
print(f"تعداد پیش‌بینی‌ها: {len(submission)}")
display(submission.head())

In [ ]:
# پیش‌بینی برای داده‌های آزمون
y_pred = best_model.predict(X_test)

# تبدیل به برچسب‌های اصلی
predictions = label_encoder.inverse_transform(y_pred)

# ایجاد DataFrame خروجی
submission = pd.DataFrame({
    'category': predictions
})

# ذخیره فایل
submission.to_csv('submission.csv', index=False)
print(f"تعداد پیش‌بینی‌ها: {len(submission)}")
submission.head()

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
<b>سلول جواب‌ساز</b>
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    برای ساخته‌شدن فایل <code>result.zip</code> سلول زیر را اجرا کنید. توجه داشته باشید که پیش از اجرای سلول زیر تغییرات اعمال شده در نت‌بوک را ذخیره کرده باشید (<code>ctrl+s</code>) تا در صورت نیاز به پشتیبانی امکان بررسی کد شما وجود داشته باشد.
</font>
</p>

In [ ]:
import zipfile
import joblib
import os

if not os.path.exists(os.path.join(os.getcwd(), 'text_categorization.ipynb')):
    %notebook -e text_categorization.ipynb


def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

submission.to_csv('submission.csv', index=False)
file_names = ['text_categorization.ipynb', 'submission.csv']
compress(file_names)

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
💭 اضافه: ابرِ کلمات (Word Cloud)
</font>
</h2>

<center>
<img src="wordcloud.png">
</center>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
یکی از کتابخانه‌های بسیار جالب مرتبط با متن در پایتون، <a href="https://github.com/amueller/word_cloud"><code>WordCloud</code></a> نام دارد. این کتابخانه به شما کمک می‌کند تا ابری از پرتکرارترین توکن‌های موجود در یک مجموعه‌متن را به شکلی زیبا به تصویر بکشید. خوشبختانه نسخه‌ی فارسی این کتابخانه نیز وجود دارد که می‌توانید از <a href="https://github.com/alihoseiny/word_cloud_fa">این لینک</a> به صفحه‌ی گیت‌هاب آن مراجعه کنید. حتی می‌توانید به‌صورت دلخواه تصویری را تعیین کنید تا نمایش نهایی توکن‌ها تداعی‌کننده‌ی تصویر موردنظر شما باشد.
</font>
</p>

In [ ]:
cloud_text = ''
for text in X_train:
    cloud_text += text + ' '

In [ ]:
from wordcloud_fa import WordCloudFa

wc = WordCloudFa(width=1200, height=800, persian_normalize=True, include_numbers=False, max_words=120, background_color='white', min_font_size=10, max_font_size=100)
word_cloud = wc.generate(cloud_text)
image = word_cloud.to_image()
image.show()
image.save('wordcloud.png')

<h4 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
<b>راهنمایی</b>
</font>
</h4>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    ۱. از تکنیک n-gram کمک بگیرید.
    <br>
    ۲. توازن مجموعه‌داده را بررسی کنید.
    <br>
    ۳. در پیش‌پردازش خود می‌توانید حذف حروف اضافه و اعداد، حذف کلمات توقف، نرمال‌سازی و... را آزمایش کنید.
</font>
</p>